# Employee Skills
Building the employee skills mapping using O*NET software skills and the real employee job roles.

In [1]:
import pandas as pd
import os

attr = pd.read_csv('../data/processed/employee_attrition_processed.csv')
role_master = pd.read_csv('../data/processed/role_master.csv')

# Build a job role -> required software skills mapping from O*NET
soft_df = pd.read_csv('../data/raw/software_skills.csv')
print(f'Software skills dataset: {soft_df.shape}')
print(f'Columns: {list(soft_df.columns)}')
print(soft_df.head(3))

Software skills dataset: (31821, 7)
Columns: ['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']
  O*NET-SOC Code             Title Workplace Example Element ID  \
0     11-1011.00  Chief Executives     Adobe Acrobat    2.E.5.b   
1     11-1011.00  Chief Executives   AdSense Tracker    2.E.6.f   
2     11-1011.00  Chief Executives    Atlassian JIRA    2.E.5.a   

                                  Element Name Hot Technology In Demand  
0                 Document management software              Y         N  
1  Data base user interface and query software              N         N  
2                    Content workflow software              Y         N  


In [2]:
# Map O*NET Title -> top hot technologies
# Filter to In Demand / Hot Technology items only
hot = soft_df[soft_df.get('In Demand', soft_df.get('Hot Technology', pd.Series(dtype=str))).astype(str).str.lower() == 'yes'] if 'In Demand' in soft_df.columns or 'Hot Technology' in soft_df.columns else soft_df

# Build title -> [software list] map
if 'Workplace Example' in soft_df.columns:
    role_soft_map = soft_df.groupby('Title')['Workplace Example'].apply(lambda x: list(x.unique())[:4]).to_dict()
elif 'Software' in soft_df.columns:
    role_soft_map = soft_df.groupby('Title')['Software'].apply(lambda x: list(x.unique())[:4]).to_dict()
else:
    role_soft_map = {}

# Assign current skills to each employee based on their JobRole
def get_employee_skills(jobrole):
    skills = role_soft_map.get(jobrole, [])
    if not skills:
        for title in role_soft_map:
            if any(word in title.lower() for word in jobrole.lower().split() if len(word) > 4):
                skills = role_soft_map[title][:4]
                break
    return ', '.join(skills) if skills else 'General Office Tools'

attr['current_skills'] = attr['JobRole'].apply(get_employee_skills)

emp_skills = attr[['EmployeeID', 'JobRole', 'current_skills']].copy()
emp_skills.to_csv('../data/processed/employee_skills_current.csv', index=False)

print(f'Employee skills mapping saved: {emp_skills.shape}')
print(emp_skills.head(5).to_string(index=False))

Employee skills mapping saved: (500, 3)
 EmployeeID         JobRole                                                                                current_skills
          1         Auditor 1099 ProsSoftware, Abacus Tax, Accountants Templates JAZZ-It!, Accounting compliance software
          2 Sales Executive             Adobe Creative Cloud software, Adobe Illustrator, Adobe InDesign, Adobe Photoshop
          3        Helpdesk                                                                          General Office Tools
          4    HR Executive                   Adobe Acrobat, AdSense Tracker, Atlassian JIRA, Blackbaud The Raiser's Edge
          5 Account Manager 1099 ProsSoftware, Abacus Tax, Accountants Templates JAZZ-It!, Accounting compliance software
